In [1]:
import os
import sys
import time

# ── environment ───────────────────────────────────────────────────────
os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"]           = r"D:\hadoop"
os.environ["PATH"]                  += os.pathsep + r"D:\hadoop\bin"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

# ── start spark ───────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("Airline_Processing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("=" * 55)
print("SPARK PROCESSING — AIRLINE PIPELINE")
print("=" * 55)
print(f"✅ Spark version   : {spark.version}")
print(f"✅ Spark UI        : {spark.sparkContext.uiWebUrl}")
print(f"✅ CPU cores       : {spark.sparkContext.defaultParallelism}")
print("=" * 55)
print("🚀 Spark ready!")

SPARK PROCESSING — AIRLINE PIPELINE
✅ Spark version   : 4.1.1
✅ Spark UI        : http://view-localhost:4041
✅ CPU cores       : 12
🚀 Spark ready!


In [2]:
PARQUET_PATH = r"D:\airline_pipeline\data\processed\flights_clean.parquet"

print("=" * 55)
print("LOADING CLEAN PARQUET DATA")
print("=" * 55)

start = time.time()
df = spark.read.parquet(PARQUET_PATH)
duration = round(time.time() - start, 2)

print(f"✅ Parquet loaded in   : {duration} seconds")
print(f"✅ Total columns       : {len(df.columns)}")
print(f"✅ Partitioned by      : Year + Month")
print(f"✅ Years available     : 2022, 2023, 2024, 2025")
print(f"\n📋 Columns available:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")
print("=" * 55)
print("🚀 Data ready for analysis!")

LOADING CLEAN PARQUET DATA
✅ Parquet loaded in   : 3.4 seconds
✅ Total columns       : 34
✅ Partitioned by      : Year + Month
✅ Years available     : 2022, 2023, 2024, 2025

📋 Columns available:
    1. Quarter
    2. DayofMonth
    3. DayOfWeek
    4. FlightDate
    5. Reporting_Airline
    6. Flight_Number_Reporting_Airline
    7. Origin
    8. OriginCityName
    9. OriginState
   10. Dest
   11. DestCityName
   12. DestState
   13. CRSDepTime
   14. DepTime
   15. DepDelay
   16. DepDelayMinutes
   17. CRSArrTime
   18. ArrTime
   19. ArrDelay
   20. ArrDelayMinutes
   21. Cancelled
   22. CancellationCode
   23. Diverted
   24. Distance
   25. CarrierDelay
   26. WeatherDelay
   27. NASDelay
   28. SecurityDelay
   29. LateAircraftDelay
   30. IsDelayed
   31. IsCancelled
   32. TotalDelay
   33. Year
   34. Month
🚀 Data ready for analysis!


Q1 → Which airline delays the most?
     (grouped by airline, avg delay, cancel rate)

Q2 → Which routes are most unreliable?
     (Origin → Dest pairs with highest delays)

Q3 → What causes delays the most?
     (Weather vs Carrier vs NAS vs Security)

Q4 → Which months are worst for flying?
     (seasonal patterns across 2022-2025)

Q5 → Window functions — rolling delay trends
     (7-day rolling average per airline)
     ← This is the VG advanced requirement!

Q6 → Which airports are busiest and most delayed?
     (airport-level analysis with joins)
     ← This covers the JOIN requirement!

In [3]:
print("=" * 55)
print("Q1 — AIRLINE DELAY PERFORMANCE")
print("=" * 55)

start = time.time()

airline_performance = df \
    .groupBy("Reporting_Airline") \
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.sum("IsCancelled") / F.count("*") * 100, 2).alias("CancelRate_pct"),
        F.round(F.sum("IsDelayed") / F.count("*") * 100, 2).alias("DelayRate_pct"),
        F.round(F.avg("TotalDelay"), 2).alias("AvgTotalDelay")
    ) \
    .filter(F.col("TotalFlights") > 10000) \
    .orderBy(F.desc("AvgArrDelay"))

duration = round(time.time() - start, 2)

print(f"⏱️  Computed in : {duration} seconds")
print(f"\n{'Airline':<10} {'Flights':>10} {'AvgArrDelay':>12} {'DelayRate%':>11} {'CancelRate%':>12}")
print("-" * 60)

results = airline_performance.collect()
for row in results:
    print(f"  {row['Reporting_Airline']:<8} "
          f"{row['TotalFlights']:>10,} "
          f"{row['AvgArrDelay']:>12.1f} "
          f"{row['DelayRate_pct']:>11.1f} "
          f"{row['CancelRate_pct']:>12.1f}")

print("=" * 55)
print(f"✅ Total airlines analysed : {len(results)}")
print("✅ Q1 complete!")

Q1 — AIRLINE DELAY PERFORMANCE
⏱️  Computed in : 0.87 seconds

Airline       Flights  AvgArrDelay  DelayRate%  CancelRate%
------------------------------------------------------------
  F9          722,989         15.8        28.3          2.2
  B6          999,045         14.3        27.5          2.3
  G4          469,009         13.7        25.2          1.6
  AA        3,692,776         12.7        23.2          1.9
  NK          940,089          9.8        23.7          2.0
  YV          114,779          9.0        18.5          3.3
  OH          864,795          8.9        20.0          2.7
  HA          306,741          7.1        18.5          1.0
  OO        2,924,404          6.3        17.2          1.4
  UA        2,848,542          6.1        19.3          1.4
  WN        5,438,843          5.9        20.7          1.5
  MQ        1,029,387          4.9        18.3          1.8
  QX           88,791          4.5        15.6          1.7
  AS          944,033          3.7  

In [5]:
print("=" * 55)
print("Q1 — SPARK TABLE VIEW")
print("=" * 55)

Q1 — SPARK TABLE VIEW


In [6]:
print("=" * 55)
print("Q2 — MOST UNRELIABLE ROUTES")
print("=" * 55)

start = time.time()

route_performance = df \
    .withColumn("Route", F.concat(
        F.col("Origin"), F.lit(" → "), F.col("Dest")
    )) \
    .groupBy("Route", "Origin", "Dest") \
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.sum("IsDelayed") / F.count("*") * 100, 2).alias("DelayRate_pct"),
        F.round(F.sum("IsCancelled") / F.count("*") * 100, 2).alias("CancelRate_pct"),
        F.round(F.avg("Distance"), 0).alias("AvgDistance_miles")
    ) \
    .filter(F.col("TotalFlights") > 500) \
    .orderBy(F.desc("AvgArrDelay"))

duration = round(time.time() - start, 2)

print(f"⏱️  Computed in : {duration} seconds")
print(f"\n🔴 TOP 15 WORST ROUTES (highest avg arrival delay):")
print(f"\n{'Route':<15} {'Flights':>8} {'AvgDelay':>9} {'DelayRate%':>11} {'Distance':>9}")
print("-" * 60)

worst_routes = route_performance.limit(15).collect()
for row in worst_routes:
    print(f"  {row['Route']:<13} "
          f"{row['TotalFlights']:>8,} "
          f"{row['AvgArrDelay']:>9.1f} "
          f"{row['DelayRate_pct']:>11.1f} "
          f"{row['AvgDistance_miles']:>9.0f}")

print()
print(f"\n🟢 TOP 10 BEST ROUTES (lowest avg arrival delay):")
print(f"\n{'Route':<15} {'Flights':>8} {'AvgDelay':>9} {'DelayRate%':>11} {'Distance':>9}")
print("-" * 60)

best_routes = route_performance.orderBy(F.asc("AvgArrDelay")).limit(10).collect()
for row in best_routes:
    print(f"  {row['Route']:<13} "
          f"{row['TotalFlights']:>8,} "
          f"{row['AvgArrDelay']:>9.1f} "
          f"{row['DelayRate_pct']:>11.1f} "
          f"{row['AvgDistance_miles']:>9.0f}")

total_routes = route_performance.count()
print("=" * 55)
print(f"✅ Total routes analysed : {total_routes:,}")
print("✅ Q2 complete!")

Q2 — MOST UNRELIABLE ROUTES
⏱️  Computed in : 0.22 seconds

🔴 TOP 15 WORST ROUTES (highest avg arrival delay):

Route            Flights  AvgDelay  DelayRate%  Distance
------------------------------------------------------------
  RNO → JFK          680      66.1        43.8      2411
  SNA → PVU        1,184      51.2        37.2       565
  AUS → HNL          522      44.0        49.0      3764
  DFW → ALB          726      43.8        48.2      1435
  HTS → PIE          682      41.2        42.8       721
  LEX → SFB          835      40.4        41.3       667
  HTS → SFB          561      39.7        46.7       665
  MFR → LAS          638      39.4        37.5       600
  USA → FLL          825      37.6        38.8       643
  EGE → MIA          532      37.2        40.0      1810
  TVC → DFW          546      36.9        41.9      1022
  ABQ → JFK          951      36.6        38.7      1826
  DFW → MDT        1,430      36.3        39.0      1231
  EYW → PHL        1,612     

In [7]:
print("=" * 55)
print("Q3 — DELAY ROOT CAUSE ANALYSIS")
print("=" * 55)

start = time.time()

# ── total minutes per delay cause ────────────────────────────────────
delay_causes = df \
    .filter(F.col("ArrDelay") > 0) \
    .agg(
        F.round(F.sum("CarrierDelay"), 0).alias("Carrier"),
        F.round(F.sum("WeatherDelay"), 0).alias("Weather"),
        F.round(F.sum("NASDelay"), 0).alias("NAS"),
        F.round(F.sum("SecurityDelay"), 0).alias("Security"),
        F.round(F.sum("LateAircraftDelay"), 0).alias("LateAircraft"),
        F.count("*").alias("TotalDelayedFlights")
    )

result = delay_causes.collect()[0]
duration = round(time.time() - start, 2)

total_mins = (result["Carrier"] + result["Weather"] +
              result["NAS"] + result["Security"] +
              result["LateAircraft"])

print(f"⏱️  Computed in : {duration} seconds")
print(f"\n📊 DELAY CAUSE BREAKDOWN (total minutes 2022-2025):")
print(f"\n{'Cause':<20} {'Total Mins':>14} {'Percentage':>11}")
print("-" * 50)

causes = [
    ("Carrier",      result["Carrier"]),
    ("Late Aircraft", result["LateAircraft"]),
    ("NAS (Traffic)", result["NAS"]),
    ("Weather",      result["Weather"]),
    ("Security",     result["Security"]),
]

for cause, mins in sorted(causes, key=lambda x: x[1], reverse=True):
    pct  = (mins / total_mins) * 100
    bar  = "█" * int(pct / 2)
    print(f"  {cause:<18} {mins:>14,.0f}  {pct:>8.1f}%  {bar}")

print("-" * 50)
print(f"  {'TOTAL':<18} {total_mins:>14,.0f}  {'100.0%':>9}")
print(f"\n✅ Total delayed flights : {result['TotalDelayedFlights']:,}")

# ── by year breakdown ─────────────────────────────────────────────────
print(f"\n📅 DELAY CAUSES BY YEAR:")
print(f"\n{'Year':<6} {'Carrier%':>9} {'Weather%':>9} {'NAS%':>9} {'LateAC%':>9}")
print("-" * 50)

yearly = df \
    .filter(F.col("ArrDelay") > 0) \
    .groupBy("Year") \
    .agg(
        F.round(F.sum("CarrierDelay"), 0).alias("Carrier"),
        F.round(F.sum("WeatherDelay"), 0).alias("Weather"),
        F.round(F.sum("NASDelay"), 0).alias("NAS"),
        F.round(F.sum("LateAircraftDelay"), 0).alias("LateAC"),
    ) \
    .orderBy("Year") \
    .collect()

for row in yearly:
    total = row["Carrier"] + row["Weather"] + row["NAS"] + row["LateAC"]
    print(f"  {row['Year']}  "
          f"{row['Carrier']/total*100:>8.1f}%"
          f"{row['Weather']/total*100:>9.1f}%"
          f"{row['NAS']/total*100:>9.1f}%"
          f"{row['LateAC']/total*100:>9.1f}%")

print("=" * 55)
print("✅ Q3 complete!")

Q3 — DELAY ROOT CAUSE ANALYSIS
⏱️  Computed in : 11.96 seconds

📊 DELAY CAUSE BREAKDOWN (total minutes 2022-2025):

Cause                    Total Mins  Percentage
--------------------------------------------------
  Late Aircraft         154,305,876      39.3%  ███████████████████
  Carrier               140,037,362      35.7%  █████████████████
  NAS (Traffic)          74,787,266      19.1%  █████████
  Weather                22,726,786       5.8%  ██
  Security                  717,885       0.2%  
--------------------------------------------------
  TOTAL                 392,575,175     100.0%

✅ Total delayed flights : 9,775,084

📅 DELAY CAUSES BY YEAR:

Year    Carrier%  Weather%      NAS%   LateAC%
--------------------------------------------------
  2022      39.9%      5.6%     16.8%     37.7%
  2023      36.5%      5.2%     18.1%     40.1%
  2024      34.6%      6.0%     18.9%     40.5%
  2025      32.4%      6.4%     22.2%     39.0%
✅ Q3 complete!


In [9]:
print("=" * 55)
print("Q4 — SEASONAL DELAY PATTERNS")
print("=" * 55)

start = time.time()

month_names = {
    1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr",
    5:"May", 6:"Jun", 7:"Jul", 8:"Aug",
    9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"
}

monthly = df \
    .groupBy("Year", "Month") \
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.sum("IsDelayed") / F.count("*") * 100, 2).alias("DelayRate_pct"),
        F.round(F.sum("IsCancelled") / F.count("*") * 100, 2).alias("CancelRate_pct"),
    ) \
    .orderBy("Year", "Month") \
    .collect()

duration = round(time.time() - start, 2)
print(f"⏱️  Computed in : {duration} seconds")

print(f"\n{'Month':<6} {'2022':>8} {'2023':>8} {'2024':>8} {'2025':>8}  Bar")
print("-" * 65)

# ── group by month ────────────────────────────────────────────────────
from collections import defaultdict
by_month = defaultdict(dict)
for row in monthly:
    by_month[row["Month"]][row["Year"]] = row["AvgArrDelay"]

for m in range(1, 13):
    name  = month_names[m]
    y2022 = by_month[m].get(2022, 0) or 0
    y2023 = by_month[m].get(2023, 0) or 0
    y2024 = by_month[m].get(2024, 0) or 0
    y2025 = by_month[m].get(2025, 0) or 0
    avg   = (y2022 + y2023 + y2024 + y2025) / 4
    bar   = "█" * max(0, int(avg / 2))
    print(f"  {name:<4} {y2022:>8.1f} {y2023:>8.1f} {y2024:>8.1f} {y2025:>8.1f}  {bar}")

# ── worst and best months overall ────────────────────────────────────
overall_monthly = df \
    .groupBy("Month") \
    .agg(
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.sum("IsDelayed") / F.count("*") * 100, 2).alias("DelayRate_pct"),
        F.count("*").alias("TotalFlights")
    ) \
    .orderBy(F.desc("AvgArrDelay")) \
    .collect()

print(f"\n🔴 WORST months to fly:")
for row in overall_monthly[:3]:
    print(f"   {month_names[row['Month']]} → "
          f"avg delay {row['AvgArrDelay']} mins, "
          f"{row['DelayRate_pct']}% flights delayed")

print(f"\n🟢 BEST months to fly:")
for row in overall_monthly[-3:]:
    print(f"   {month_names[row['Month']]} → "
          f"avg delay {row['AvgArrDelay']} mins, "
          f"{row['DelayRate_pct']}% flights delayed")

print("=" * 55)
print("✅ Q4 complete!")

Q4 — SEASONAL DELAY PATTERNS
⏱️  Computed in : 6.21 seconds

Month      2022     2023     2024     2025  Bar
-----------------------------------------------------------------
  Jan       3.7      7.6      9.9      3.6  ███
  Feb       4.5      4.1      0.6      5.6  █
  Mar       7.0      8.9      6.4      5.4  ███
  Apr       8.4      8.9      5.3      5.6  ███
  May       7.2      3.8     13.8     10.1  ████
  Jun      10.3     14.0     11.6     15.3  ██████
  Jul       9.7     16.2     17.5     16.8  ███████
  Aug       8.5      8.3     10.3      9.1  ████
  Sep       2.4      5.3      1.0      2.3  █
  Oct       2.1      1.2     -1.3      5.5  
  Nov       4.7     -1.2     -0.3      6.2  █
  Dec      13.0      0.5      7.0      0.0  ██

🔴 WORST months to fly:
   Jul → avg delay 15.11 mins, 26.24% flights delayed
   Jun → avg delay 12.83 mins, 24.76% flights delayed
   Aug → avg delay 9.06 mins, 21.39% flights delayed

🟢 BEST months to fly:
   Sep → avg delay 2.73 mins, 16.04% fligh

In [11]:
print("=" * 55)
print("Q5 — WINDOW FUNCTIONS (ROLLING DELAY TRENDS)")
print("=" * 55)

start = time.time()

# ── monthly avg delay per airline ────────────────────────────────────
monthly_airline = df \
    .groupBy("Year", "Month", "Reporting_Airline") \
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgDelay"),
        F.round(F.sum("IsDelayed") / F.count("*") * 100, 2).alias("DelayRate_pct")
    ) \
    .withColumn("YearMonth",
        F.col("Year") * 100 + F.col("Month"))

# ── window spec — rolling 3 month average per airline ─────────────────
window_spec = Window \
    .partitionBy("Reporting_Airline") \
    .orderBy("YearMonth") \
    .rowsBetween(-2, 0)

# ── apply window functions ────────────────────────────────────────────
df_window = monthly_airline \
    .withColumn("Rolling3MonthAvgDelay",
        F.round(F.avg("AvgDelay").over(window_spec), 2)) \
    .withColumn("Rolling3MonthDelayRate",
        F.round(F.avg("DelayRate_pct").over(window_spec), 2)) \
    .withColumn("RankByDelay",
        F.rank().over(
            Window.partitionBy("Year", "Month")
            .orderBy(F.desc("AvgDelay"))
        )) \
    .withColumn("CumulativeFlights",
        F.sum("TotalFlights").over(
            Window.partitionBy("Reporting_Airline")
            .orderBy("YearMonth")
            .rowsBetween(Window.unboundedPreceding, 0)
        ))

duration = round(time.time() - start, 2)
print(f"⏱️  Computed in : {duration} seconds")

# ── show rolling trend for top 5 airlines ────────────────────────────
print(f"\n📊 ROLLING 3-MONTH AVG DELAY — Top 5 Airlines:")
print(f"\n{'Airline':<10} {'Year':<6} {'Month':<7} {'AvgDelay':>9} {'3M_Rolling':>11} {'Rank':>5}")
print("-" * 55)

top_airlines = ["AA", "DL", "UA", "WN", "F9"]

rolling_results = df_window \
    .filter(F.col("Reporting_Airline").isin(top_airlines)) \
    .filter(F.col("Year") == 2024) \
    .orderBy("Reporting_Airline", "Month") \
    .select("Reporting_Airline", "Year", "Month",
            "AvgDelay", "Rolling3MonthAvgDelay", "RankByDelay") \
    .collect()

for row in rolling_results:
    print(f"  {row['Reporting_Airline']:<8} "
          f"{row['Year']:<6} "
          f"{row['Month']:<7} "
          f"{row['AvgDelay']:>9.1f} "
          f"{row['Rolling3MonthAvgDelay']:>11.1f} "
          f"{row['RankByDelay']:>5}")

# ── lag analysis — improvement year over year ─────────────────────────
print(f"\n📈 YEAR OVER YEAR IMPROVEMENT (using LAG window function):")
print(f"\n{'Airline':<10} {'2023_Delay':>11} {'2024_Delay':>11} {'Change':>8}")
print("-" * 45)

yearly_airline = df \
    .groupBy("Year", "Reporting_Airline") \
    .agg(F.round(F.avg("ArrDelay"), 2).alias("AvgDelay")) \
    .withColumn("PrevYearDelay",
        F.lag("AvgDelay", 1).over(
            Window.partitionBy("Reporting_Airline")
            .orderBy("Year")
        )) \
    .withColumn("YoY_Change",
        F.round(F.col("AvgDelay") - F.col("PrevYearDelay"), 2)) \
    .filter(F.col("Year") == 2024) \
    .filter(F.col("Reporting_Airline").isin(top_airlines)) \
    .orderBy("YoY_Change") \
    .collect()

for row in yearly_airline:
    arrow = "📈 worse" if row["YoY_Change"] > 0 else "📉 better"
    print(f"  {row['Reporting_Airline']:<8} "
          f"{row['PrevYearDelay']:>11.1f} "
          f"{row['AvgDelay']:>11.1f} "
          f"{row['YoY_Change']:>8.1f}  {arrow}")

print("=" * 55)
print("✅ Window functions used:")
print("   ✅ Rolling 3-month average  (rowsBetween)")
print("   ✅ Rank by delay per month  (rank)")
print("   ✅ Cumulative flights       (unboundedPreceding)")
print("   ✅ Year-over-year lag       (lag)")
print("✅ Q5 complete")

Q5 — WINDOW FUNCTIONS (ROLLING DELAY TRENDS)
⏱️  Computed in : 1.08 seconds

📊 ROLLING 3-MONTH AVG DELAY — Top 5 Airlines:

Airline    Year   Month    AvgDelay  3M_Rolling  Rank
-------------------------------------------------------
  AA       2024   1            18.8         8.3     1
  AA       2024   2             4.8         9.2     5
  AA       2024   3            16.3        13.3     3
  AA       2024   4            12.6        11.2     2
  AA       2024   5            29.0        19.3     1
  AA       2024   6            22.4        21.4     2
  AA       2024   7            30.9        27.4     1
  AA       2024   8            20.9        24.7     2
  AA       2024   9             5.6        19.1     3
  AA       2024   10            2.0         9.5     4
  AA       2024   11            2.5         3.4     3
  AA       2024   12           10.6         5.1     4
  DL       2024   1             4.3        -2.8    14
  DL       2024   2            -4.2        -2.1    13
  DL      

In [13]:
print("=" * 55)
print("Q6 — AIRPORT ANALYSIS WITH JOINS")
print("=" * 55)

start = time.time()

# ── busiest departure airports ────────────────────────────────────────
origin_stats = df \
    .groupBy("Origin", "OriginCityName", "OriginState") \
    .agg(
        F.count("*").alias("DepartureFlights"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.sum("IsDelayed") / F.count("*") * 100, 2).alias("DelayRate_pct"),
        F.round(F.sum("IsCancelled") / F.count("*") * 100, 2).alias("CancelRate_pct")
    ) \
    .withColumnRenamed("Origin", "Airport") \
    .withColumnRenamed("OriginCityName", "CityName") \
    .withColumnRenamed("OriginState", "State")

# ── busiest arrival airports ──────────────────────────────────────────
dest_stats = df \
    .groupBy("Dest", "DestCityName", "DestState") \
    .agg(
        F.count("*").alias("ArrivalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay")
    ) \
    .withColumnRenamed("Dest", "Airport") \
    .withColumnRenamed("DestCityName", "CityName") \
    .withColumnRenamed("DestState", "State")

# ── JOIN departures and arrivals on airport code ──────────────────────
airport_joined = origin_stats.join(
    dest_stats,
    on="Airport",
    how="inner"
) \
    .withColumn("TotalFlights",
        F.col("DepartureFlights") + F.col("ArrivalFlights")) \
    .withColumn("AvgOverallDelay",
        F.round((F.col("AvgDepDelay") + F.col("AvgArrDelay")) / 2, 2)) \
    .filter(F.col("TotalFlights") > 100000) \
    .orderBy(F.desc("TotalFlights"))

duration = round(time.time() - start, 2)
print(f"⏱️  Computed in : {duration} seconds")

# ── top 15 busiest airports ───────────────────────────────────────────
print(f"\n✈️  TOP 15 BUSIEST AIRPORTS:")
print(f"\n{'Airport':<8} {'City':<25} {'TotalFlights':>13} {'AvgDelay':>9} {'DelayRate%':>11}")
print("-" * 72)

busiest = airport_joined \
    .orderBy(F.desc("TotalFlights")) \
    .limit(15) \
    .collect()

for row in busiest:
    city = row["CityName"][:23] if row["CityName"] else "N/A"
    print(f"  {row['Airport']:<6} "
          f"{city:<25} "
          f"{row['TotalFlights']:>13,} "
          f"{row['AvgOverallDelay']:>9.1f} "
          f"{row['DelayRate_pct']:>11.1f}%")

# ── most delayed airports ─────────────────────────────────────────────
print(f"\n🔴 TOP 10 MOST DELAYED AIRPORTS:")
print(f"\n{'Airport':<8} {'City':<25} {'AvgDelay':>9} {'DelayRate%':>11}")
print("-" * 58)

most_delayed = airport_joined \
    .orderBy(F.desc("AvgOverallDelay")) \
    .limit(10) \
    .collect()

for row in most_delayed:
    city = row["CityName"][:23] if row["CityName"] else "N/A"
    print(f"  {row['Airport']:<6} "
          f"{city:<25} "
          f"{row['AvgOverallDelay']:>9.1f} "
          f"{row['DelayRate_pct']:>11.1f}%")

# ── best airports ─────────────────────────────────────────────────────
print(f"\n🟢 TOP 10 BEST AIRPORTS (least delayed):")
print(f"\n{'Airport':<8} {'City':<25} {'AvgDelay':>9} {'DelayRate%':>11}")
print("-" * 58)

best_airports = airport_joined \
    .orderBy(F.asc("AvgOverallDelay")) \
    .limit(10) \
    .collect()

for row in best_airports:
    city = row["CityName"][:23] if row["CityName"] else "N/A"
    print(f"  {row['Airport']:<6} "
          f"{city:<25} "
          f"{row['AvgOverallDelay']:>9.1f} "
          f"{row['DelayRate_pct']:>11.1f}%")

total_airports = airport_joined.count()
print("=" * 55)
print(f"✅ Total airports analysed  : {total_airports}")
print(f"✅ Join type used           : INNER JOIN")
print(f"✅ Joined on                : Airport code")
print(f"✅ Left table               : Origin stats")
print(f"✅ Right table              : Destination stats")
print("✅ Q6 complete")

Q6 — AIRPORT ANALYSIS WITH JOINS
⏱️  Computed in : 0.81 seconds

✈️  TOP 15 BUSIEST AIRPORTS:

Airport  City                       TotalFlights  AvgDelay  DelayRate%
------------------------------------------------------------------------
  ATL    Atlanta, GA                   2,569,819       7.9        19.2%
  DFW    Dallas/Fort Worth, TX         2,331,198      14.3        24.5%
  DEN    Denver, CO                    2,321,465      11.0        23.8%
  ORD    Chicago, IL                   2,197,406      11.0        22.0%
  CLT    Charlotte, NC                 1,569,792      11.5        22.8%
  LAX    Los Angeles, CA               1,507,991       7.3        17.2%
  LAS    Las Vegas, NV                 1,449,800      10.9        22.9%
  PHX    Phoenix, AZ                   1,429,060       8.8        18.6%
  SEA    Seattle, WA                   1,306,675       5.6        19.2%
  LGA    New York, NY                  1,239,883       9.9        19.7%
  MCO    Orlando, FL                   1,

In [16]:
import os

RESULTS_DIR = r"D:\airline_pipeline\data\processed\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 55)
print("SAVING ALL RESULTS")
print("=" * 55)

# ── save Q1 airline performance ───────────────────────────────────────
airline_performance.write \
    .mode("overwrite") \
    .parquet(RESULTS_DIR + r"\airline_performance.parquet")
print("✅ Q1 saved → airline_performance.parquet")

# ── save Q2 route performance ─────────────────────────────────────────
route_performance.write \
    .mode("overwrite") \
    .parquet(RESULTS_DIR + r"\route_performance.parquet")
print("✅ Q2 saved → route_performance.parquet")

# ── save Q4 monthly trends ────────────────────────────────────────────
overall_monthly_df = df \
    .groupBy("Year", "Month") \
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.sum("IsDelayed") / F.count("*") * 100, 2).alias("DelayRate_pct"),
        F.round(F.sum("IsCancelled") / F.count("*") * 100, 2).alias("CancelRate_pct")
    ) \
    .orderBy("Year", "Month")

overall_monthly_df.write \
    .mode("overwrite") \
    .parquet(RESULTS_DIR + r"\monthly_trends.parquet")
print("✅ Q4 saved → monthly_trends.parquet")

# ── fix duplicate columns before saving Q6 ───────────────────────────
airport_clean = airport_joined \
    .select(
        "Airport",
        origin_stats["CityName"].alias("CityName"),
        origin_stats["State"].alias("State"),
        "DepartureFlights",
        "ArrivalFlights",
        "TotalFlights",
        "AvgDepDelay",
        "AvgArrDelay",
        "AvgOverallDelay",
        "DelayRate_pct",
        "CancelRate_pct"
    )

airport_clean.write \
    .mode("overwrite") \
    .parquet(RESULTS_DIR + r"\airport_analysis.parquet")
print("✅ Q6 saved → airport_analysis.parquet")

print("=" * 55)
print("\n🎉 SPARK PROCESSING COMPLETE!")
print("=" * 55)
print(f"\n{'Question':<8} {'Analysis':<35} {'Status'}")
print("-" * 60)
print(f"  Q1     {'Airline delay performance':<35} ✅ Done")
print(f"  Q2     {'Route reliability analysis':<35} ✅ Done")
print(f"  Q3     {'Delay root cause breakdown':<35} ✅ Done")
print(f"  Q4     {'Seasonal monthly patterns':<35} ✅ Done")
print(f"  Q5     {'Window functions + rolling avg':<35} ✅ Done ")
print(f"  Q6     {'Airport analysis with joins':<35} ✅ Done ")
print("-" * 60)
print(f"\n{'Operation':<30} {'Coverage'}")
print("-" * 50)
print(f"  {'Filter':<28} ✅ Q1, Q2, Q3")
print(f"  {'GroupBy + Aggregate':<28} ✅ Q1, Q2, Q3, Q4")
print(f"  {'Join':<28} ✅ Q6 (inner join)")
print(f"  {'Window Functions':<28} ✅ Q5 (rolling, rank, lag)")
print(f"  {'OrderBy':<28} ✅ All questions")
print(f"  {'Complex aggregations':<28} ✅ Q3 (multi-column)")
print("=" * 55)
print(f"\n📁 Results saved to:")
print(f"   {RESULTS_DIR}")
print(f"\n🚀 Ready for Step 7 — Dask Processing!")

SAVING ALL RESULTS
✅ Q1 saved → airline_performance.parquet
✅ Q2 saved → route_performance.parquet
✅ Q4 saved → monthly_trends.parquet
✅ Q6 saved → airport_analysis.parquet

🎉 SPARK PROCESSING COMPLETE!

Question Analysis                            Status
------------------------------------------------------------
  Q1     Airline delay performance           ✅ Done
  Q2     Route reliability analysis          ✅ Done
  Q3     Delay root cause breakdown          ✅ Done
  Q4     Seasonal monthly patterns           ✅ Done
  Q5     Window functions + rolling avg      ✅ Done (VG!)
  Q6     Airport analysis with joins         ✅ Done (VG!)
------------------------------------------------------------

Operation                      Coverage
--------------------------------------------------
  Filter                       ✅ Q1, Q2, Q3
  GroupBy + Aggregate          ✅ Q1, Q2, Q3, Q4
  Join                         ✅ Q6 (inner join)
  Window Functions             ✅ Q5 (rolling, rank, lag)
  OrderB

In [17]:
spark.stop()
print("=" * 55)
print("✅ Spark session stopped cleanly!")
print("✅ All results saved to Parquet")
print("✅ Step 6 — Spark Processing COMPLETE!")
print("=" * 55)
print("\n📊 What we proved:")
print("   11.4 GB dataset processed successfully")
print("   6 analytical questions answered")
print("   All VG operations demonstrated")
print("   Results saved for report use")
print("\n🚀 Next → Step 7: Dask Processing")

✅ Spark session stopped cleanly!
✅ All results saved to Parquet
✅ Step 6 — Spark Processing COMPLETE!

📊 What we proved:
   11.4 GB dataset processed successfully
   6 analytical questions answered
   All VG operations demonstrated
   Results saved for report use

🚀 Next → Step 7: Dask Processing
